# exp-007: Gemini Judge 활성화 — 휴리스틱 → Gemini 재라벨

- **목적:** 휴리스틱 Judge의 *순환 편향* 제거. 동일 모델·동일 100쌍/500쌍을 Gemini API (`gemini-3-flash-preview`) 로 재라벨하여 (a) Spearman/Pearson 비교 (b) Model C 우위 축소 검증.
- **차별성 축:** ④ GT 부재 + ⑤ End-to-End LLM
- **데이터:** 
  - 입력: `benchmark_pairs_100.csv` + `benchmark_labeled_100_{A,B,C,D,E}.csv` (총 600 unique pairs)
  - user 프로필: `user_profiles.csv` (Gemini profile fields)
  - JD 프로필: `jd_profiles_sample1000.csv`
- **메트릭:** Spearman(휴리스틱 ↔ Gemini), Pearson, Cohen's κ + 동일 모델 NDCG 차이
- **비용 추정:** ~600쌍 × 2-3초/쌍 ≈ 20-30분, $1-3 USD
- **관련 위키:** dual-encoder-진화-실험계획 exp-007, embedding-model-evaluation-skill §5
- **작성일:** 2026-05-18

## ⚠️ 실행 보류 사유

현재 `exp-005-baseline-full-data.ipynb`가 Gemini API를 통해 999명 user profile을 생성 중 (User 단계 ~35%, 약 2시간 남음). 동시 Gemini 호출은 rate limit 충돌 위험. **exp-005 완료 후 실행** 권장.

TODO 실행 시:
1. exp-005 user profile 단계 완료 확인
2. JD profile 단계 진입 전에 일시 정지 (Cell 5 후)
3. 본 노트북 실행 (600쌍 재라벨)
4. exp-005 JD 단계 재개

In [ ]:
# ⚠️ 실행 보류 — exp-005 Gemini 호출 충돌 우려
# 아래 코드는 작성 완료, 실행만 미루는 상태

import pandas as pd
import numpy as np
from pathlib import Path
import json
import os
import hashlib
from dotenv import load_dotenv

load_dotenv('output/.env')

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-007-gemini-judge')
CACHE_DIR = Path('raw/data/gemini_cache/judge_v2')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_MODEL = os.getenv('GEMINI_JUDGE_MODEL', 'gemini-3-flash-preview')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
assert GEMINI_API_KEY, 'GEMINI_API_KEY 필요'

In [ ]:
# 600 unique pair 수집 (single 100 + 5관점 500 → set으로 unique)
single = pd.read_csv(DATA / 'benchmark_labeled_100.csv', encoding='utf-8-sig')
single.columns = [c.lstrip('\ufeff') for c in single.columns]
single['eval_category'] = 'SINGLE'

persp_dfs = []
for p in 'ABCDE':
    df = pd.read_csv(DATA / f'benchmark_labeled_100_{p}.csv', encoding='utf-8-sig')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    df['eval_category'] = p
    persp_dfs.append(df)

all_pairs = pd.concat([single[['userId', 'job_id', 'eval_category', 'judge_relevance']]] + 
                       [d[['userId', 'job_id', 'eval_category', 'judge_relevance']] for d in persp_dfs],
                       ignore_index=True)
print(f'총 행: {len(all_pairs)}')
print(f'unique (user, JD) 쌍: {len(all_pairs.drop_duplicates(["userId", "job_id"]))}')

In [ ]:
# Gemini Judge prompt
JUDGE_SCHEMA = {
    'type': 'object',
    'properties': {
        'relevance': {'type': 'integer', 'enum': [0, 1, 2, 3, 4]},
        'rationale': {'type': 'string'},
        'matched_evidence': {'type': 'array', 'items': {'type': 'string'}},
        'missing_requirements': {'type': 'array', 'items': {'type': 'string'}},
        'confidence': {'type': 'number'},
    },
    'required': ['relevance', 'rationale', 'confidence']
}
JUDGE_PROMPT = '''당신은 한국어 채용 매칭 평가자입니다. 아래 User Profile과 JD Profile을 보고 0~4 점수로 평가하세요.
- 4: 완벽한 매칭 (직무·산업·스킬·자소서 모두 일치)
- 3: 강한 매칭 (대부분 일치, 일부 차이)
- 2: 부분 매칭 (절반 일치)
- 1: 약한 매칭 (1-2개 요소만 일치)
- 0: 매칭 없음

User Profile:
{user_profile}

JD Profile:
{jd_profile}

JSON 응답:'''

# TODO: google.genai 호출 + sha256 캐시 + concurrent
# 본 셀은 실제 실행 시 활성화
print('⏸️ 실행 보류 중 — exp-005 종료 후 활성화')

In [ ]:
# (실행 시) Gemini Judge 호출 → 600쌍 재라벨
# (실행 시) 비교 메트릭: Spearman, Pearson, Cohen's κ
# (실행 시) NDCG@10 재계산 비교
# placeholder
print('실행 시 결과: raw/experiments/exp-007-gemini-judge/{gemini_labels.csv, comparison_metrics.json}')